 # AI Ruby on Rails Code Review

 This agent helps to review pull requests in minutes, it suggest best practices, coding convention, security vectors and linters. 

# Install dependencies

In [1]:
%pip install -r requirements.txt


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Imports

In [3]:
from src.github import download_pr, clone_pr_repository
from src.parser import parse_diff
from src.rag import load_vector_db
from src.rubocop import run_rubocop, format_offenses
from src.reviewer import review

# Load DB

In [4]:
db = load_vector_db()

print(f"RAG database is ready.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11262.96it/s]


RAG database is ready.


# Download Pull Request

In [9]:
PR_URL = "https://github.com/renzodiaz/notes-api/pull/4"

diff = download_pr(PR_URL)

print(f"Downloaded {len(diff)} characters.")

Downloaded 5625 characters.


# Parse the Pull Request

In [10]:
parsed = parse_diff(diff)

print("Production Ruby files:")

for file in parsed["production_files"]:
    print("-", file["filename"])

print()
print("Test files:")

for file in parsed["test_files"]:
    print("-", file["filename"])

print(parsed["added_ruby_code"])

Production Ruby files:
- app/controllers/api/v1/auth_controller.rb
- app/controllers/api/v1/secure_controller.rb
- app/models/user.rb
- config/routes.rb
- db/migrate/20260808154603_create_users.rb
- db/schema.rb

Test files:
- test/models/user_test.rb

FILE: app/controllers/api/v1/auth_controller.rb

HUNK:
@@ -0,0 +1,13 @@

ADDED CODE:
module Api::V1
    class AuthController < SecureController
        def login
            user = User.find_by(email: params[:email])

            if user && user.authenticate(params[:password])
                render json: { user: user }, status: :ok
            end

            render json: { error: "Invalid email or password" }, status: :unauthorized
        end
    end
end

REMOVED CODE:


CONTEXT:
\ No newline at end of file


FILE: app/controllers/api/v1/secure_controller.rb

HUNK:
@@ -0,0 +1,4 @@

ADDED CODE:
module Api::V1
    class SecureController < ApplicationController
    end
end

REMOVED CODE:


CONTEXT:
\ No newline at end of file


FILE: ap

# Clone branch PR repository

In [11]:
repo_path = clone_pr_repository(
    PR_URL
)

print(
    "Repository cloned to:",
    repo_path,
)

Repository cloned to: /var/folders/b1/d_q83c6s5t98lzfhlg1w5vhw0000gn/T/review_ai_jzxf58jp


# Run Rubocop

In [12]:
ruby_files = [
    file["filename"]
    for file in parsed["ruby_files"]
]

offenses = run_rubocop(
    project_path=repo_path,
    files=ruby_files,
)

print(
    f"RuboCop found {len(offenses)} offenses."
)

RuboCop found 3 offenses.


In [13]:
rubocop_context = format_offenses(
    offenses
)

print(rubocop_context)


File: app/controllers/api/v1/auth_controller.rb
Line: 13
Severity: convention
Cop: Layout/TrailingEmptyLines
Message: Final newline missing.


File: app/controllers/api/v1/secure_controller.rb
Line: 4
Severity: convention
Cop: Layout/TrailingEmptyLines
Message: Final newline missing.


File: config/routes.rb
Line: 10
Severity: convention
Cop: Style/StringLiterals
Message: Prefer double-quoted strings unless you need single quotes to avoid extra backslashes for escaping.



# Review & Print

In [14]:
review_result = review(
    changed_code=parsed["added_ruby_code"],
    db=db,
    rubocop_context=rubocop_context,
)

print(review_result)

# Overall Review

## Summary

This PR adds a basic API login flow using `has_secure_password`, introduces a `User` model and migration, and wires a `/api/v1/login` route. It is generally on the right track, but there are a couple of important correctness and security-related issues in the controller and schema that should be addressed before merge.

## Issues

### [High] Login action renders success and failure responses unconditionally

**Category:** Security / Maintainability / Rails  
**File:** `app/controllers/api/v1/auth_controller.rb`  
**Line:** 4-10  

**Explanation:**  
The `login` action currently does:

```ruby
if user && user.authenticate(params[:password])
  render json: { user: user }, status: :ok
end

render json: { error: "Invalid email or password" }, status: :unauthorized
```

Because there is no `return` after the հաջող `render`, the unauthorized response will also be executed after a successful login. In Rails, this will typically raise a `DoubleRenderError` or othe